In [0]:
# =============================================================================
# 1. CREACIÓN DE ESTRUCTURA BRONZE - Tablas MANAGED (compatible Free Edition)
# =============================================================================

dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "1. Ambiente")

env = dbutils.widgets.get("env")
catalog = f"{env}_fraud"
schema_bronze = f"{catalog}.bronze"

print(f"Creando estructura MANAGED en catálogo: {catalog} | schema: bronze\n")

# Asegúrate de que el schema bronze exista (si no, créalo primero)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_bronze}")

# 1. clientes_bronze
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_bronze}.clientes_bronze (
    id_cliente               BIGINT,
    nombre                   STRING,
    edad_actual              INT,
    edad_jubilacion          INT,
    ano_nacimiento           INT,
    mes_nacimiento           INT,
    genero                   STRING,
    direccion                STRING,
    apartamento              STRING,
    ciudad                   STRING,
    estado                   STRING,
    codigo_postal            STRING,
    latitud                  DOUBLE,
    longitud                 DOUBLE,
    ingreso_per_capita_zip   DOUBLE,
    ingreso_anual            DOUBLE,
    deuda_total              DOUBLE,
    puntaje_fico             INT,
    num_tarjetas             INT,
    
    ingestion_timestamp      TIMESTAMP,
    source_file              STRING,
    process_date             STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
)
""")

print("→ clientes_bronze creada (managed)")

# 2. tarjetas_bronze
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_bronze}.tarjetas_bronze (
    id_tarjeta               BIGINT,
    id_cliente               BIGINT,
    indice_tarjeta           INT,
    marca_tarjeta            STRING,
    tipo_tarjeta             STRING,
    numero_tarjeta           STRING,
    fecha_expiracion         STRING,
    cvv                      INT,
    tiene_chip               STRING,
    tarjetas_emitidas        INT,
    limite_credito           DOUBLE,
    fecha_apertura           STRING,
    ano_ultimo_pin           INT,
    en_dark_web              STRING,
    
    ingestion_timestamp      TIMESTAMP,
    source_file              STRING,
    process_date             STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
)
""")

print("→ tarjetas_bronze creada (managed)")

# 3. transacciones_bronze
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_bronze}.transacciones_bronze (
    id_cliente               BIGINT,
    indice_tarjeta           INT,
    ano                      INT,
    mes                      INT,
    dia                      INT,
    hora                     STRING,
    monto                    DOUBLE,
    metodo_pago              STRING,
    nombre_comerciante       STRING,
    ciudad_comerciante       STRING,
    estado_comerciante       STRING,
    zip_comerciante          DOUBLE,
    codigo_mcc               INT,
    errores                  STRING,
    es_fraude                STRING,
    
    ingestion_timestamp      TIMESTAMP,
    source_file              STRING,
    process_date             STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
)
""")

print("→ transacciones_bronze creada (managed)")

# 4. catalogo_mcc_bronze
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {schema_bronze}.catalogo_mcc_bronze (
    codigo_mcc               INT,
    descripcion              STRING,
    descripcion_combinada    STRING,
    descripcion_usda         STRING,
    descripcion_irs          STRING,
    reportable_irs           STRING,
    
    ingestion_timestamp      TIMESTAMP,
    source_file              STRING,
    process_date             STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5',
    'delta.columnMapping.mode' = 'name'
)
""")

print("→ catalogo_mcc_bronze creada (managed)")

print("\n✅ Todas las tablas bronze creadas como MANAGED")
print("Ahora puedes ingestar datos con .saveAsTable() sin problemas en Free Edition")

# Verificación rápida
display(spark.sql(f"SHOW TABLES IN {schema_bronze}"))